# 实验任务一: 预训练模型

------
### **1. 使用GPU训练模型**
    
在PyTorch中，可以使用以下代码来检测当前环境是否有可用的GPU：

In [1]:
import torch

# 检查是否有可用的GPU
if torch.cuda.is_available():
    print(f"CUDA is available. Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name(torch.cuda.current_device())}")
else:
    print("CUDA is not available. Using CPU.")

CUDA is available. Number of GPUs: 1
Current device: 0
Device name: NVIDIA GeForce RTX 3070 Ti Laptop GPU


如果显示'CUDA is not available. Using CPU.'请确认启动的环境是否正确或者尝试重新安装pytorch或者与助教联系。

把模型放到GPU上的代码示例。定义模型后，通过model = model.to(device)把模型放到GPU上。

In [ ]:
# 检查是否有可用的GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 创建模型
model = SimpleModel()

# 将模型放到GPU（如果可用）
model = model.to(device)

把数据放到GPU上的代码示例。由于模型在GPU上，所以数据也必须在GPU上才能送入模型。通过inputs = inputs.to(device)把input放到GPU上。

值得说明的是由于模型的输出也在GPU上，所以标签也需要放到GPU上以便于计算损失，通过labels = labels.to(device)。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# 训练示例
num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    for inputs, labels in train_loader:
        # 将数据放到GPU（如果可用）
        inputs, labels = inputs.to(device), labels.to(device)

        # 前向传播
        outputs = model(inputs)

通过上述过程，我们可以把数据和模型都放到GPU上从而加速训练。

你可以使用以下命令查看是否使用了GPU并且观察的GPU利用率：

watch -n 5 nvidia-smi

这个命令会每5秒（-n 5）更新一次NVIDIA GPU的状态信息。

### **2. 了解预训练语言模型**
    
下面我们以BERT为例，用的bert-base-uncased版本进行实验。我们首先用AutoModel和AutoTokenizer加载模型和分词器。分词器是把文本的每个词元映射到对应的索引，以便于BERT的embedding层完成索引到嵌入的映射。


完整代码如下：

In [4]:
import torch
from transformers import AutoModel, AutoTokenizer

# 指定模型名称
model_name = 'bert-base-uncased'

# 读取模型对应的tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 载入模型
model = AutoModel.from_pretrained(model_name)

# 输入文本
input_text = "Here is some text to encode"

# 通过tokenizer把文本变成 token_id
input_ids = tokenizer.encode(input_text, add_special_tokens=True)
print(input_ids)

# 转换为Tensor
input_ids = torch.tensor([input_ids])

# 获得BERT的输出
with torch.no_grad():
    output = model(input_ids)

# 获得BERT模型最后一个隐层结果
output_hidden_states = output.last_hidden_state
output_hidden_states.shape

/root/miniconda3/envs/pytorch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[101, 2182, 2003, 2070, 3793, 2000, 4372, 16044, 102]


torch.Size([1, 9, 768])

分词（tokenizer）的过程会在文本的头尾添加特殊token，即会在文本的开头加入词元[CLS]并且在文本的结尾加入词元[SEP]。你可以调整input_text和设置add_special_tokens=False，观察到这两个词元分别被编码为101和102。

除此之外，由于批处理过程需要一个批次中文本长度相同，因此额外引入了padding。所以，我们需要使用了attention_mask屏蔽这些padding token，不让其参与自注意力的计算。

最终的输出是文本中所有词元的隐藏状态（hidden states）。

我们可以用model.named_parameters(): 观察模型的所有参数及其形状，完整代码如下：

In [5]:
import torch
from transformers import AutoModel, AutoTokenizer

# 指定模型名称
model_name = 'bert-base-uncased'

# 读取模型对应的tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 载入模型
model = AutoModel.from_pretrained(model_name)

# 打印模型所有参数的名称和形状
for name, param in model.named_parameters():
    print(f"Parameter Name: {name}, Shape: {param.shape}")

Parameter Name: embeddings.word_embeddings.weight, Shape: torch.Size([30522, 768])
Parameter Name: embeddings.position_embeddings.weight, Shape: torch.Size([512, 768])
Parameter Name: embeddings.token_type_embeddings.weight, Shape: torch.Size([2, 768])
Parameter Name: embeddings.LayerNorm.weight, Shape: torch.Size([768])
Parameter Name: embeddings.LayerNorm.bias, Shape: torch.Size([768])
Parameter Name: encoder.layer.0.attention.self.query.weight, Shape: torch.Size([768, 768])
Parameter Name: encoder.layer.0.attention.self.query.bias, Shape: torch.Size([768])
Parameter Name: encoder.layer.0.attention.self.key.weight, Shape: torch.Size([768, 768])
Parameter Name: encoder.layer.0.attention.self.key.bias, Shape: torch.Size([768])
Parameter Name: encoder.layer.0.attention.self.value.weight, Shape: torch.Size([768, 768])
Parameter Name: encoder.layer.0.attention.self.value.bias, Shape: torch.Size([768])
Parameter Name: encoder.layer.0.attention.output.dense.weight, Shape: torch.Size([768, 7

### **3. 使用预训练模型进行文本分类**
可能需要安装transformers包


In [ ]:
pip install transformers

在本章节中，你将基于上面的BERT代码和AG NEWS数据集进行基于预训练模型BERT的文本分类。你将完善下述代码同时探索多种句子聚合方式对结果的影响，其中句子聚合方式指的是从词嵌入中得到句子嵌入的过程。需要探索的句子聚合方式包括：

1. 直接使用[CLS]的嵌入表示当做句子嵌入。
2. 使用mean-pooling平均一个句子中的所有词元得到嵌入
3. 使用注意力机制给每个词元分配一个权重，通过加权求和的方式得到嵌入。你可以使用任意注意力机制计算。

代码部分：

In [15]:
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# 配置参数
AGGREGATION_METHOD = 'attn'  # 可选 'cls', 'mean', 'attn'
MAX_LENGTH = 128
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
EPOCHS = 3

df = pd.read_csv("/root/DL-lab/lab5/ag/train.csv")
df.columns = ["label", "title", "description"]
df["text"] = df["title"] + " " + df["description"]
df["label"] = df["label"] - 1
train_texts, train_labels = df["text"].tolist(), df["label"].tolist()
number = int(0.3 * len(train_texts))
train_texts, train_labels = train_texts[: number], train_labels[: number]

df = pd.read_csv("/root/DL-lab/lab5/ag/test.csv")
df.columns = ["label", "title", "description"]
df["text"] = df["title"] + " " + df["description"]
df["label"] = df["label"] - 1
test_texts, test_labels = df["text"].tolist(), df["label"].tolist()

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

class AGNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=50):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            text, truncation=True, padding="max_length", max_length=self.max_length, return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }
        

train_dataset = AGNewsDataset(train_texts, train_labels, tokenizer)
test_dataset = AGNewsDataset(test_texts, test_labels, tokenizer)

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False)

class BERTClassifier(nn.Module):
    def __init__(self, model_name, num_labels, aggregation='cls'):
        super(BERTClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.aggregation = aggregation
        self.num_labels = num_labels
        
        # 注意力聚合专用层
        if aggregation == 'attn':
            self.attention = nn.Sequential(
                nn.Linear(self.bert.config.hidden_size, 128),
                nn.Tanh(),
                nn.Linear(128, 1)
            )
            
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        
        if self.aggregation == 'cls':
            # [CLS] 聚合
            pooled = outputs.pooler_output
            
        elif self.aggregation == 'mean':
            # 均值池化
            last_hidden = outputs.last_hidden_state
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
            sum_embeddings = torch.sum(last_hidden * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            pooled = sum_embeddings / sum_mask
            
        elif self.aggregation == 'attn':
            # 注意力池化
            last_hidden = outputs.last_hidden_state
            attention_weights = self.attention(last_hidden).squeeze(-1)
            attention_weights = attention_weights.masked_fill(attention_mask == 0, -1e9)
            attention_weights = torch.softmax(attention_weights, dim=-1)
            pooled = torch.sum(last_hidden * attention_weights.unsqueeze(-1), dim=1)
            
        else:
            raise ValueError("不支持的聚合方法")
            
        return self.classifier(pooled)

# 初始化模型
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BERTClassifier(model_name, num_labels=4, aggregation=AGGREGATION_METHOD).to(device)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    loop = tqdm(train_dataloader, desc=f"Epoch {epoch+1}")

    for batch in loop:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()

        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        loop.set_postfix(loss=loss.item())
    
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_dataloader):.4f}")

    
    model.eval()
    preds, true_labels = [], []

    with torch.no_grad():
        for batch in test_dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"]
            true_labels.extend(labels.numpy())

            logits = model(input_ids, attention_mask)
            preds.extend(logits.argmax(dim=1).cpu().numpy())


    acc = accuracy_score(true_labels, preds)
    print(f"Aggregation: {AGGREGATION_METHOD}, Epoch {epoch+1}, Loss: {total_loss/len(train_dataloader):.4f}, Acc: {acc:.4f}")

/root/miniconda3/envs/pytorch/lib/python3.10/site-packages/transformers/optimization.py:640: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
Epoch 1: 100%|██████████| 2250/2250 [05:56<00:00,  6.31it/s, loss=0.149] 


Epoch 1, Loss: 0.2654
Aggregation: attn, Epoch 1, Loss: 0.2654, Acc: 0.9191


Epoch 2: 100%|██████████| 2250/2250 [06:30<00:00,  5.75it/s, loss=0.134]  


Epoch 2, Loss: 0.1472
Aggregation: attn, Epoch 2, Loss: 0.1472, Acc: 0.9250


Epoch 3: 100%|██████████| 2250/2250 [06:44<00:00,  5.56it/s, loss=0.445]  


Epoch 3, Loss: 0.0934
Aggregation: attn, Epoch 3, Loss: 0.0934, Acc: 0.9236


训练笔记：你如果觉得训练速度慢，可以尝试增大batch size，不过注意不要炸显存。

思考题1：你觉得以上三种得到句子嵌入的方案，哪种效果会最好，哪种效果会最差？为什么？

思考题2：如果一个文档包括多个句子，我们需要获得其中每个句子的嵌入表示。那么，我们应该怎么利用BERT得到每个句子的嵌入？